#  EMIT data analysis for EMIT Project

**[NASA’s Earth Surface Mineral Dust Source Investigation (EMIT) mission](https://earth.jpl.nasa.gov/emit/)** 


Strong winds lift mineral dust from Earth’s desert and dryland regions and carry the dust particles great distances. Depending on its color and composition, mineral dust can affect Earth’s environment in different ways. Dark particles absorb sunlight and heat our planet, while light-colored particles can have a cooling effect.

Minerals containing iron tend to be dark, and those with clay tend to be lighter in color. To improve global science models and understand how dust heats and cools the Earth’s environment, scientists need to know the dust’s mineral composition as it blows around Earth.

For the first time, the NASA Earth Surface Mineral Dust Source Investigation (EMIT) mission will comprehensively measure the mineral composition of Earth’s dust source regions to help scientists understand how they heat or cool our planet. EMIT’s science objectives are specifically focused on better understanding this heating and cooling effect, which is called radiative forcing. 

* The first objective is to deliver a new improved assessment of the heating and cooling effects of mineral dust in the Earth’s atmosphere. 
* The second objective is to predict how future climate scenarios might change the amount and type of mineral dust emitted into the Earth’s atmosphere.


In [ ]:
import xarray as xr
import netCDF4,os,re,copy,glob
import numpy as np
import matplotlib.pyplot as plt
from EMIT_lib import *
import pickle 
import cartopy
import sys,os
import cartopy.crs as crs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import awscli, s3fs, boto3
import requests
import xarray as xr
import hvplot.xarray
import holoviews as hv
from AWS_Utils import get_NASA_creds

## define EMIT data class

### EMIT instrument
EMIT uses an advanced imaging spectrometer instrument that measures a spectrum for every point in the image. Sunlight reflected from minerals on the Earth’s surface is imaged by a telescope and spectrometer system onto a detector area array that is sensitive from the visible to short wavelength infrared portion of the electromagnetic spectrum. The EMIT dust source minerals have distinct spectral signatures in this wavelength range. Each column of the detector array records the spectrum for a sample of 1240 cross-track EMIT instrument swath. The along-track dimension of the image is built up with the forward motion of the ISS. The imaging spectrometer measured image cubes are calibrated to spectral radiance, then atmospherically corrected and analyzed to determine the mineral composition that is required to achieve the EMIT science objectives.

### Key EMIT Properties

* F-number: F/1.8 
* Cross-track FOV: 11°
* IFOV (cross-track x along-track): 155 x 71 μrad
* Focal length: 193.5 mm
* Entrance pupil aperture: 110 mm
* **Spectral Range: 380 – 2500 nm**
* **Spectral Sampling: 7.4 nm**


### EMIT data product

The EMIT mission will provide a suite of data products for all areas successfully measured below the International Space Station. These data and derived products will be used to achieve the EMIT science objectives, reducing the uncertainty in mineral dust radiative forcing. All measurements will be delivered to the NASA Land Processes Distributed Active Archive Center and made available to the broader community for the additional Earth science investigations it can enable.

The EMIT data products for each stage of analysis from Level 1b to Level 4 are described in a series of Algorithm Theoretical Basis Documents (ATBDs). These describe the theoretical foundations for each stage of the analysis, define the quantities in the data files, and provide guidance on interpretation. Uncertainty predictions are distributed with all products.

**as of 04/25/2023 only L1 and L2a products are avaiable**

| Data Product | Description | 
| --- | --- |
| Level 0 | Raw collected telemetry|
| Level 1a | Reconstructed, depacketized, uncompressed data, time referenced, annotated with ancillary information reassembled into scenes|
| Level 1b | Level 1a data processed to sensor units including geolocation and observation geometry information|
| Level 2a | Surface reflectance derived by screening clouds and correction for atmospheric effects|
| Level 2b | Mineralogy derived from fitting reflectance spectra, screening for non-mineralogical components|
| Level 3 | Gridded map of mineral composition aggregated from level 2b with uncertainties and quality flags|
|  Level 4 |Earth System Model runs to address science objectives|

#### import the AWS credientials 
below is a method to request credentials from each NASA DAAC center 
adopted from workshop
https://nasa-openscapes.github.io/2021-Cloud-Workshop-AGU/how-tos/Earthdata_Cloud__Single_File__Direct_S3_Access_NetCDF4_Example.html

In [ ]:
lpdaac_creds = get_NASA_creds('lpdaac')
fs_s3 = s3fs.S3FileSystem(anon=False, 
                          key=lpdaac_creds['accessKeyId'], 
                          secret=lpdaac_creds['secretAccessKey'], 
                          token=lpdaac_creds['sessionToken'])

In [ ]:
# Define a S3 client
s3Client = boto3.client(
    's3',
    aws_access_key_id=lpdaac_creds['accessKeyId'],
    aws_secret_access_key=lpdaac_creds['secretAccessKey'],
    aws_session_token=lpdaac_creds['sessionToken'])

#### check the bucket for EMIT data

In [ ]:
RAD_bucket_prefix='s3://lp-prod-protected/EMITL1BRAD.001/'
RAD_file_prefix='EMIT_L1B_RAD_001_'
RFL_bucket_prefix='s3://lp-prod-protected/EMITL2ARFL.001/'
RFL_file_prefix='EMIT_L2A_RFL_001_'
MIN_bucket_prefix='s3://lp-prod-protected/EMITL2BMIN.001/'
MIN_file_prefix='EMIT_L2B_MIN_001_'

In [ ]:
time_stamp = '20231227T091713'

In [ ]:
RAD_bucket = os.path.join(RAD_bucket_prefix, RAD_file_prefix+time_stamp+'_*')
print(RAD_bucket)
RAD_subdir = fs_s3.glob(RAD_bucket)
RAD_files =  fs_s3.glob(os.path.join(RAD_subdir[0],'*.nc'))
print(RAD_files)

### def a function to read EMIT data in AWS

In [ ]:
def read_single_EMIT_granule_data(unique_time_stamp,products2read):
    '''
    Inputs: 
    unique_time_stamp" '20231227T091713' the unique time stamp to track EMIT data
    products2read: list of products to read e.g., ['RAD','RFL','MIN']
    Outputs:
    RAD: Radiance data
    RFL: Surface reflectance data
    
    '''
    response = requests.get('https://data.lpdaac.earthdatacloud.nasa.gov/s3credentials')
    temp_creds_req = response.json()
    fs_s3 = s3fs.S3FileSystem(anon=False, 
                          key=temp_creds_req['accessKeyId'], 
                          secret=temp_creds_req['secretAccessKey'], 
                          token=temp_creds_req['sessionToken'])
    RAD_bucket_prefix='s3://lp-prod-protected/EMITL1BRAD.001/'
    RAD_file_prefix='EMIT_L1B_RAD_001_'
    RFL_bucket_prefix='s3://lp-prod-protected/EMITL2ARFL.001/'
    RFL_file_prefix='EMIT_L2A_RFL_001_'
    MIN_bucket_prefix='s3://lp-prod-protected/EMITL2BMIN.001/'
    MIN_file_prefix='EMIT_L2B_MIN_001_'
    EMIT_granule_data={}
    for d in products2read:
        if d=='RAD' or d=='OBS':
            bucket = os.path.join(RAD_bucket_prefix, RAD_file_prefix+unique_time_stamp+'_*')
            #print(bucket)
            subdir = fs_s3.glob(bucket)
            product_files =  fs_s3.glob(os.path.join(subdir[0],'*'+d+'*.nc'))
            if len(product_files) ==1:
                s3_file_obj = fs_s3.open('s3://'+product_files[0], mode='rb')
                print('reading...',product_files[0])
                #print(s3_file_obj.group())
                EMIT_granule_data[d]=xr.open_dataset(s3_file_obj, engine='h5netcdf')
               
                EMIT_granule_data[d+'_location']=xr.open_dataset(s3_file_obj,
                                                   group="/location",
                                                   engine='h5netcdf')

                EMIT_granule_data[d+'_sensor_band_parameters']=xr.open_dataset(s3_file_obj,
                                                                   group="/sensor_band_parameters",
                                                                   engine='h5netcdf')
        elif d =='RFL':
            bucket = os.path.join(RFL_bucket_prefix, RFL_file_prefix+unique_time_stamp+'_*')
            #print(bucket)
            subdir = fs_s3.glob(bucket)
            product_files =  fs_s3.glob(os.path.join(subdir[0],'*'+d+'*.nc'))
            for p in product_files:
                if 'RFLUNCERT' in p: continue
                s3_file_obj = fs_s3.open('s3://'+p, mode='rb')
                print('reading...',p)
                EMIT_granule_data[d]=xr.open_dataset(s3_file_obj, engine='h5netcdf')
            #print(product_files)
            # s3_file_obj = fs_s3.open('s3://'+ssh_files[0], mode='rb')
            #
    return EMIT_granule_data
        

In [ ]:
case_stamp='20231227T091713'
#case_stamp='20240124T143039'
EMIT_granule_data_dict=read_single_EMIT_granule_data(case_stamp,['RAD','OBS','RFL'])

In [ ]:
EMIT_lon0=(EMIT_granule_data_dict["RAD_location"].lon.values.mean())
EMIT_lat0=(EMIT_granule_data_dict["RAD_location"].lat.values.mean())
print(EMIT_lon0,EMIT_lat0)

#### plot the RGB image

In [ ]:

R_idx,G_idx,B_idx=36,22,10
#R_idx,G_idx,B_idx=200,22,4

print(EMIT_granule_data_dict['RAD_sensor_band_parameters'].wavelengths[[R_idx,G_idx,B_idx]].values)
      #.sensor_band_parameters['wavelengths'][[R_idx,G_idx,B_idx]])
#print(np.diff(RAD.sensor_band_parameters['wavelengths']))
scale_factor= np.max(np.array([EMIT_granule_data_dict['RAD'].radiance.values[:,:,R_idx].max(),
                               EMIT_granule_data_dict['RAD'].radiance.values[:,:,G_idx].max(),
                               EMIT_granule_data_dict['RAD'].radiance.values[:,:,B_idx].max()]))
print(scale_factor)
RGB=np.stack([EMIT_granule_data_dict['RAD'].radiance.values[:,:,R_idx]/scale_factor*255,
              EMIT_granule_data_dict['RAD'].radiance.values[:,:,G_idx]/scale_factor*255,
              EMIT_granule_data_dict['RAD'].radiance.values[:,:,B_idx]/scale_factor*255],axis=2)
RGB_image=np.array(RGB, dtype=np.uint8)
print(RGB.shape)
fig,ax=plt.subplots(figsize=[6,6],tight_layout=True)
ax.imshow(RGB_image,origin='lower')

print(RGB_image.min(),RGB_image.max())
# for px,py in zip(pixel_xindex,pixel_yindex):
#     ax.plot([px],[py],'o',c='r')
fig.savefig('EMIT_granlue'+case_stamp+'_RGB.png',dpi=300)

In [ ]:
EMIT_granule_data_dict['RAD'].radiance.values[:,:,R_idx].shape

## read MODIS data in AWS

In [ ]:
temp_creds_req = get_temp_creds('laadsdaac')
fs_s3 = s3fs.S3FileSystem(anon=False, 
                          key=temp_creds_req['accessKeyId'], 
                          secret=temp_creds_req['secretAccessKey'], 
                          token=temp_creds_req['sessionToken'])

In [ ]:
fs_s3.ls('s3://prod-lads/MYD021KM/MYD021KM.A2024024.1420.061.2024025184206.hdf')

In [ ]:
MODIS_buckets={'MOD021KM':'s3://prod-lads/MOD021KM',
               'MYD021KM':'s3://prod-lads/MYD021KM',
               'MOD06_L2':'s3://prod-lads/MOD06_L2',
               'MYD06_L2':'s3://prod-lads/MYD06_L2'
              }

In [ ]:
RAD_subdir = fs_s3.glob(os.path.join(MODIS_buckets['MYD021KM'],'MYD021KM.A2024024.1420.061.2024025184206.hdf'))
# print(os.path.join(MODIS_buckets['MYD06_L2'],'MYD06_L2.A2024076.0555.061.*.hdf'))
# print(MODIS_buckets['MYD06_L2']+'/MYD06_L2.A2024076.0555.061.2024076174439.hdf')
# print('s3://prod-lads/MYD06_L2/MYD06_L2.A2024076.0555.061.2024076174439.hdf')
#s3://prod-lads/MYD021KM/MYD021KM.A2024024.1420.061.2024025184206.hdf

In [ ]:
print(RAD_subdir)

In [ ]:
import MODIS_lib

In [ ]:
ML1B=MODIS_lib.MOD021KM('MYD021KM.A2024024.1420.061.2024025184206.hdf',Radiance=True, Read_ems=False)
MGEO=MODIS_lib.MOD03('MYD03.A2024024.1420.061.2024025155900.hdf')

In [ ]:
print(MGEO.lat.min(),MGEO.lat.max(),EMIT_lat0)
print(MGEO.lon.min(),MGEO.lon.max(),EMIT_lon0)
MODIS_near_EMIT = np.where( (MGEO.lat >EMIT_lat0-0.05) & (MGEO.lat <EMIT_lat0+0.05) &\
                            (MGEO.lon >EMIT_lon0-0.05) & (MGEO.lon <EMIT_lon0+0.05) )
MGEO.lat[MODIS_near_EMIT]
d,x=MGEO.lat.shape
idx_flat = np.ravel_multi_index(MODIS_near_EMIT, (d,x))
idx_2d = np.unravel_index(idx_flat, (d,x))  
print(idx_2d[0])
for i in range(idx_2d[0].size):
      print(MGEO.lat[idx_2d[0][i],idx_2d[1][i]],
            MGEO.lon[idx_2d[0][i],idx_2d[1][i]],
            EMIT_lat0,EMIT_lon0)

In [ ]:
plt.figure()
MODIS_R=ML1B.bands_obs[0,:,:]
MODIS_G=ML1B.bands_obs[3,:,:]
MODIS_B=ML1B.bands_obs[2,:,:]
fig,ax=plt.subplots()
ax.imshow(MODIS_R)

fig,ax=plt.subplots()
ax.imshow(MODIS_G)

fig,ax=plt.subplots()
ax.imshow(MODIS_B)

print('max',MODIS_R.max(),MODIS_G.max(),MODIS_B.max())
scale_factor= np.max(np.array([MODIS_R.max(),MODIS_G.max(),MODIS_B.max()]))
print(scale_factor)
RGB=np.stack([MODIS_R/scale_factor*255,
              MODIS_G/scale_factor*255,
              MODIS_B/scale_factor*255],axis=2)
RGB_image=np.array(RGB, dtype=np.uint8)
fig,ax=plt.subplots()
ax.imshow(RGB_image,origin='lower')

In [ ]:


fig,ax=plt.subplots()
ax.imshow(RGB_image,origin='lower')
for i in range(idx_2d[0].size):
    print(idx_2d[0][i],idx_2d[1][i])
    ax.scatter([idx_2d[1][i],idx_2d[1][i]],
               [idx_2d[0][i],idx_2d[0][i]],c='b',marker='s')

fig,ax=plt.subplots()
for i in range(idx_2d[0].size):
    m,n=idx_2d[0][i],idx_2d[1][i]
    print(ML1B.bands_obs[0:7,m,n])
    ax.scatter(ML1B.bands_wl[0:7],ML1B.bands_obs[0:7,m,n],marker='.')
ax.set_ylim([0,120])

In [ ]:
EMIT_sp=EMIT_granule_data_dict['RAD'].radiance.values[:800,:,:].mean(axis=(0,1)) 
# 'uW/cm^2/SR/nm' -> W/m^2/SR/um'
EMIT_sp = EMIT_sp * 1e-6/(1e-4)/(1e-3) * 2.0
EMIT_wl=EMIT_granule_data_dict['RAD_sensor_band_parameters']['wavelengths'].values
fig,ax=plt.subplots()
ax.plot(EMIT_wl,EMIT_sp)
for i in range(idx_2d[0].size):
    m,n=idx_2d[0][i],idx_2d[1][i]
    ax.scatter(ML1B.bands_wl[0:7]*1000,ML1B.bands_obs[0:7,m,n],marker='s')
ax.set_ylim([0,150])

In [ ]:
EMIT_wl[35]

In [ ]:
def get_subdataset(file, varnames):
    from osgeo import gdal
    g = gdal.Open(path)
    subdatasets = g.GetSubDatasets()
    l = []
    for varname in varnames:
        l.extend(  [s[0] for s in subdatasets if varname in s[0].split(':')[-1]  ])
    return l
def read_hdf4(file, varnames):
    ld = []
    fname_list = get_subdataset(file, varnames)
    for fname in fname_list:
        myDataset = xr.open_rasterio(fname)
        ld.append(myDataset.to_dataset(name=fname.split(':')[-1]))
    return xr.merge(ld)

In [ ]:
xr.open_rasterio('s3://prod-lads/MYD021KM/MYD021KM.A2024024.1420.061.2024025184206.hdf',
                 'latitude')

### solar spectral irradiance (SSI)

To estimate the reflection of solar radiation by surface and atmsophere, from which we can estimate the aerosol radiative effects, we need to solar spectral irradiance (SSI)

These data were taken from LBLRTM 5.21
(http://www.meto.umd.edu/~bobe/LBLRTM/)
The original Kurudz [1992] data were converted to mW / (m2 nm) and 
averaged over 1nm intervals centered around the given wavelength
(BM, September 25, 2001)

 Reference:
 Kurucz, R.L., Synthetic infrared spectra, in Infrared Solar Physics,
IAU Symp. 154, edited by D.M. Rabin and J.T. Jefferies, Kluwer, Acad.,
Norwell, MA, 1992.


In [ ]:
class solar_spectrum():
    def __init__(self,data_path='./data/solar_flux/',filename='kurudz_1.0nm.dat'):
        D = np.loadtxt(data_path+filename,skiprows=11)
        self.wl=D[:,0]
        self.S0=D[:,1]
            
kurudz_1nm=solar_spectrum()
fig,ax=plt.subplots()
ax.plot(kurudz_1nm.wl,kurudz_1nm.S0)
ax.set_xlabel('wavelength [$mn$]')
ax.set_ylabel(r'spectral solar irradiance [$mW m^{-2} nm^{-1}$]')
print('solar constant=',np.trapz(kurudz_1nm.S0,kurudz_1nm.wl),r'$W m^{-2}$')
EMIT_range=(kurudz_1nm.wl>381) & (kurudz_1nm.wl<2493)
print('solar constant EMIT=',np.trapz(kurudz_1nm.S0[EMIT_range],kurudz_1nm.wl[EMIT_range]),r'$W m^{-2}$')

In [ ]:
atmos_pro = np.loadtxt('/umbc/xfs1/zzbatmos-new/users/zzbatmos/Proposal/EMIT_2023/atmos_profile.dat',
                      skiprows=2)
fig,ax=plt.subplots()
ax.plot(atmos_pro[:,2],atmos_pro[:,0])
axr=ax.twiny()
axr.plot(atmos_pro[:,4]/atmos_pro[:,3]*1e6,atmos_pro[:,0],c='r')
#axr.plot(atmos_pro[:,7]/atmos_pro[:,3]*1e6,atmos_pro[:,0],c='b')

## dust optical properties 

In [ ]:
class dust_optical_property(object):
    def __init__(self):
        D=np.loadtxt('/home/zzbatmos/zzbatmos_user/Proposal/EMIT_2023/EMIT_dust_opt_cal/opt_prop_EMIT_FennecMean_DiBiagioSW_spheriod.txt',skiprows=1)
        self.wl=D[:,0]
        self.qe=D[:,2]
        self.alb=D[:,3]
        self.gg=D[:,4]
        self.wl0_index=np.argmin(np.abs(self.wl-0.55))
    def plot_property(self):
        fig,ax=plt.subplots(3,1,figsize=[9,6],tight_layout=True)
        ax[0].plot(self.wl,self.qe)
        ax[0].set_xlabel('Wavelength [um]')
        ax[0].set_ylabel('Qe')
        
        ax[1].plot(self.wl,self.alb)
        ax[1].set_xlabel('Wavelength [um]')
        ax[1].set_ylabel('single-scattering albedo')
        
        ax[2].plot(self.wl,self.gg)
        ax[2].set_xlabel('Wavelength [um]')
        ax[2].set_ylabel('Asymmetry Factor')

DOP=dust_optical_property()
DOP.plot_property()
print(DOP.wl0_index,DOP.wl[DOP.wl0_index])

### analysis a EMIT case on 2023-02-24


In [ ]:
EMIT_data_path='/home/zzbatmos/zzbatmos_common/Data/EMIT/'

case_stamp = '20230726T103511_2320706_026' # a case with both land and ocean 
pixel_yindex,pixel_xindex = [1100,1100],[700,200] # ocean pixel and land pixel

# case_stamp =  '20230224T103018_2305507_039' #a case over land with signficant amount of dust
# pixel_yindex,pixel_xindex = [1200.1200],[479,479] 

#'20230224T103018_2305507_039' a case over land with signficant amount of dust
# case_stamp = '20220904T091058_2224706_002' # dust over ocean case
# pixel_xindex,pixel_yindex = 800,1000

#'20220816T145459_2222810_016'

# case_stamp =  '20220819T140623_2223109_022'
# pixel_xindex,pixel_yindex = 800,1000

OBS_pattern = 'EMIT_L1B_OBS'
RAD_pattern = 'EMIT_L1B_RAD'

MASK_pattern = 'EMIT_L2A_MASK'
RFL_pattern = 'EMIT_L2A_RFL_'
RFL_UNCERT_pattern = 'EMIT_L2A_RFLUNCERT'

MIN_pattern = 'EMIT_L2B_MIN_'
MIN_UNCERT_pattern = 'EMIT_L2B_MINUNCERT'


case_files=glob.glob(EMIT_data_path+'*'+case_stamp+'*.nc')
print(case_files)
for f in case_files:
    if OBS_pattern in f: 
        print('reading',f)
        OBS=EMIT_Data(f,'obs','L1B_OBS','L1B')
    elif RAD_pattern in f:
        print('reading',f)
        RAD=EMIT_Data(f,'radiance','L1B_RAD','L1B')
    elif MASK_pattern in f:
        print('reading',f)
        MASK=EMIT_Data(f,'mask','L2A_MASK','L2A')
    elif RFL_pattern in f:
        print('reading',f)
        RFL=EMIT_Data(f,'reflectance','L2A_RFL','L2A')
    elif MIN_pattern in f:
        print('reading',f)
        MIN=EMIT_Data(f,'Mineral','L2B_MIN','L2B')
            

### plot the RGB true color image of the case

EMIT has 285 spectral bands from 380 nm to 2500 nm. We select band 36, 22, 10 at the wavelengths of 648.7 nm,  544.4 nm, 455.1 nm for the R, G and B bands.  

In [ ]:
print(RAD.data.shape)
R_idx,G_idx,B_idx=36,22,10
#R_idx,G_idx,B_idx=200,22,4

print(RAD.sensor_band_parameters['wavelengths'][[R_idx,G_idx,B_idx]])
#print(np.diff(RAD.sensor_band_parameters['wavelengths']))
scale_factor= np.max(np.array([RAD.data[:,:,R_idx].max(),
                               RAD.data[:,:,G_idx].max(),
                               RAD.data[:,:,B_idx].max()]))
print(scale_factor)
RGB=np.stack([RAD.data[:,:,R_idx]/scale_factor*255,
              RAD.data[:,:,G_idx]/scale_factor*255,
              RAD.data[:,:,B_idx]/scale_factor*255],axis=2)
RGB_image=np.array(RGB, dtype=np.uint8)
print(RGB.shape)
fig,ax=plt.subplots(figsize=[6,6],tight_layout=True)
ax.imshow(RGB_image,origin='lower')

print(RGB_image.min(),RGB_image.max())
for px,py in zip(pixel_xindex,pixel_yindex):
    ax.plot([px],[py],'o',c='r')
fig.savefig('EMIT_granlue'+case_stamp+'_RGB.png',dpi=300)

In [ ]:
fig,ax=plt.subplots()
ax.imshow(MASK.data[:,:,5],origin='lower')
print(OBS.data[:,:,2].max(),OBS.data[:,:,2].min(),OBS.data[:,:,2].mean())
print(OBS.location['lat'].mean(),OBS.location['lon'].mean(),OBS.location['lon'].shape)

In [ ]:
fig,ax=plt.subplots(2,1,figsize=[6,6],tight_layout=True)
#ax[0].plot(RAD.sensor_band_parameters['wavelengths'],RAD.data[1100,200,:],c='orange',lw=2,ls='solid',label='Land pixel')
ax[0].plot(RAD.sensor_band_parameters['wavelengths'],RAD.data[1100,700,:],c='b',lw=2,ls='dashed',label='Ocean pixel')

ax[0].set_xlabel('Wavelength [nm]',fontsize='x-large')
ax[0].set_ylabel('Radiance',fontsize='x-large') # [$\mu W \ cm^{-2} \ SR^{-1} \ nm^{-1}]$
ax[0].legend(loc=0)

ax[0].set_xticks(np.arange(350,2500,250))
land_ref = RFL.data[1100,200,:]
land_ref[land_ref<=0] = np.nan

ocean_ref = RFL.data[1100,700,:]
ocean_ref[ocean_ref<=0] = np.nan
#ax[1].plot(RAD.sensor_band_parameters['wavelengths'],land_ref,c='orange',lw=2,ls='solid',label='Land pixel')
ax[1].plot(RAD.sensor_band_parameters['wavelengths'],ocean_ref,c='b',lw=2,ls='dashed',label='Ocean pixel')

ax[1].set_xlabel('Wavelength [nm]',fontsize='x-large')
ax[1].set_ylabel('Surface Reflectance',fontsize='x-large')
ax[1].legend(loc=[0.6,0.3])
ax[1].set_xticks(np.arange(350,2500,250))
fig.savefig('radiance_reflectance_example.png',dpi=300)

In [ ]:
np.isnan(land_ref)

# plot the cloud mask, and AOD @ 550 nm.
the L2A "Mask" data contains the following masks 

0. 'Cloud flag'
1. 'Cirrus flag'
2. 'Water flag'
3. 'Spacecraft Flag'
4.  'Dilated Cloud Flag'
5. 'AOD550' 
6. 'H2O (g cm-2)'
7. 'Aggregate Flag']
 

### plot SSI scaled by solar zenith angle

In [ ]:
# domain_avg_rfl = RFL.data.mean(axis=(0,1))
# import numpy.ma as ma
# y=ma.masked_where(domain_avg_rfl <= 0, domain_avg_rfl)

fig,ax=plt.subplots(figsize=(4,2),tight_layout=True)
S0_interp = np.interp(RAD.sensor_band_parameters['wavelengths'],kurudz_1nm.wl,kurudz_1nm.S0)
#ax.plot(RAD.sensor_band_parameters['wavelengths'],S0_interp*np.cos(np.deg2rad(np.mean(OBS.data[:,:,4]))),c='m',label='solar insolation')
ax.set_xlabel('wavelength [$mn$]')
ax.set_ylabel(r'[$mW m^{-2} nm^{-1}$]')
#axr=ax.twinx()
ax.plot(RAD.sensor_band_parameters['wavelengths'],RAD.data[pixel_yindex[1],pixel_xindex[1],:]*10.0*np.pi,c='orange',label='spectral irrradiance')
#ax.axvline(RAD.sensor_band_parameters['wavelengths'][G_idx])
ax.legend()
plt.savefig('spectrak_irradiance.png',dpi=300)

### plot the surface reflectance

In [ ]:
# --- domain average albedo
domain_avg_rfl = RFL.data[pixel_yindex[1],pixel_xindex[1],:]
import numpy.ma as ma
y=ma.masked_where(domain_avg_rfl <= 0, domain_avg_rfl)
S0_interp = np.interp(RAD.sensor_band_parameters['wavelengths'],kurudz_1nm.wl,kurudz_1nm.S0)
reflected_rad_no_atmos= y * S0_interp
domain_avg_spec = RAD.data[pixel_yindex[1],pixel_xindex[1],:]*10.0 # convert "uW/cm^2/SR/nm" to "mW/m^2/SR/nm"

print(y.shape)

fig,ax=plt.subplots(figsize=(4,2),tight_layout=True)
ax.plot(RAD.sensor_band_parameters['wavelengths'],y*100.0,c='g',label='surface albedo')
ax.set_xlabel('wavelength [$mn$]')
ax.set_ylabel(r'[%]')
ax.legend(loc=0)
fig.savefig('surface_albedo.png',dpi=300)


fig,ax=plt.subplots(figsize=(4,2),tight_layout=True)
S0_interp = np.interp(RAD.sensor_band_parameters['wavelengths'],kurudz_1nm.wl,kurudz_1nm.S0)
#ax.plot(RAD.sensor_band_parameters['wavelengths'],S0_interp*np.cos(np.deg2rad(np.mean(OBS.data[:,:,4]))),c='m',label='solar insolation')
ax.set_xlabel('wavelength [$mn$]')
ax.set_ylabel(r'[$mW m^{-2} nm^{-1}$]')
#axr=ax.twinx()
ax.plot(RAD.sensor_band_parameters['wavelengths'],domain_avg_spec*np.pi,c='orange',label=r'$F^{\uparrow}_{Aerosol}$')
ax.plot(RAD.sensor_band_parameters['wavelengths'],reflected_rad_no_atmos,label=r'$F^{\uparrow}_{Clean}$')

ax.legend()
fig.savefig('spectrum_comparison.png',dpi=300)
#print(reflected_rad_no_atmos)

In [ ]:
# print(E1.rad.shape)
AOD=MASK.data[:,:,5]
print('pixel AOD',AOD[pixel_yindex,pixel_xindex])
print(AOD.min(),AOD.max())
xi_max,yi_max=np.unravel_index(AOD.argmax(), AOD.shape)
xi_min,yi_min=np.unravel_index(AOD.argmin(), AOD.shape)
print(xi_max,yi_max)
print(AOD.argmax(),xi_max,yi_max)
print(AOD.argmin(),xi_min,yi_min)
fig,ax=plt.subplots()
ax.plot(RAD.data[xi_max,yi_max,:])
ax.plot(RAD.data[xi_min,yi_min,:])

axr=ax.twinx()
axr.plot(RFL.data[xi_max,yi_max,:],ls='dashed')
axr.plot(RFL.data[xi_min,yi_min,:],ls='dashed')


fig,ax=plt.subplots()
cf=ax.pcolormesh(MASK.data[:,:,5])
ax.set_title(MASK.sensor_band_parameters['mask_bands'][5])
plt.colorbar(cf,ax=ax)

fig,ax=plt.subplots(1,2,figsize=[10,4])
cf1=ax[0].pcolormesh(MIN.group_1_band_depth)
cf2=ax[1].pcolormesh(MIN.group_2_band_depth)
plt.colorbar(cf1,ax=ax[0])
plt.colorbar(cf2,ax=ax[1])


fig,ax=plt.subplots(1,2,figsize=[10,4])
cf1=ax[0].pcolormesh(MIN.group_1_mineral_id)
cf2=ax[1].pcolormesh(MIN.group_2_mineral_id)
plt.colorbar(cf1,ax=ax[0])
plt.colorbar(cf2,ax=ax[1])


# fig,ax=plt.subplots()
# cf=ax.pcolormesh(MASK.data[:,:,0])
# plt.colorbar(cf,ax=ax)
# print('pixel cloud mask',MASK.data[pixel_yindex,pixel_xindex,0])
#print(MASK.data[:,:,0].max(),MASK.data[:,:,0].min())

#print(RFL.data[xi_min,yi_min,:])
# fig,ax=plt.subplots()
# cf=ax.pcolormesh(E1.mask[:,:,1])
# plt.colorbar(cf,ax=ax)
# print(E1.mask[:,:,1].max(),E1.mask[:,:10].min())


# fig,ax=plt.subplots()
# cf=ax.pcolormesh(E1.mask[:,:,2])
# plt.colorbar(cf,ax=ax)
# print(E1.mask[:,:,2].max(),E1.mask[:,:,2].min())


In [ ]:
print(AOD[pixel_yindex,pixel_xindex])

In [ ]:
OBS.sensor_band_parameters['observation_bands']

In [ ]:
print('domain averaged ',OBS.sensor_band_parameters['observation_bands'][2],OBS.data[:,:,2].mean(axis=(0,1)))
print('domain averaged ',OBS.sensor_band_parameters['observation_bands'][4],OBS.data[:,:,4].mean(axis=(0,1)))
print('domain averaged ','relative azimuth',OBS.data[:,:,1].mean(axis=(0,1))-OBS.data[:,:,3].mean(axis=(0,1)))
solar_zenith =  OBS.data[:,:,4].mean(axis=(0,1))
sensor_zenith = OBS.data[:,:,2].mean(axis=(0,1))
relative_azimuth = np.abs(OBS.data[:,:,1].mean(axis=(0,1))-OBS.data[:,:,3].mean(axis=(0,1)))
print(solar_zenith,sensor_zenith,relative_azimuth)

In [ ]:
fig,ax=plt.subplots();
plt.plot(RAD.sensor_band_parameters['wavelengths'],RAD.data.mean(axis=(0,1)))
total_rad = np.trapz(RAD.data.mean(axis=(0,1)),RAD.sensor_band_parameters['wavelengths'])
print(total_rad/1e6*1e4*np.pi)

In [ ]:
S0 = 1200
fig,ax=plt.subplots();
RFL_spec= RFL.data.mean(axis=(0,1))
RFL_spec[RFL_spec<0] = np.nan
                        
plt.plot(RAD.sensor_band_parameters['wavelengths'],RFL_spec)

print(np.nanmean(RFL_spec)*S0)
print(RFL_spec.shape)

## generate libradtran input file for RT simulation

In [ ]:
!pwd

In [ ]:
%cd /umbc/xfs1/zzbatmos/common/Codes/libRadtran-2.0.4/examples

In [ ]:
def write_EMIT_inp(with_dust, 
                   wl_input,
                   albedo_input,
                   sza_input,
                   day_of_year_input,
                   aod_input,alb_input,gg_input,
                   atmos_file_input): 
    if with_dust:
        EMIT_input_file="EMIT_dust_wl{0:05.2f}.INP".format(wl_input)
    else:
        EMIT_input_file="EMIT_clear_wl{0:05.2f}.INP".format(wl_input)
    with open(EMIT_input_file,'w') as EMIT_inp_file:

        EMIT_inp_file.write("atmosphere_file  " + atmos_file_input + "\n")
        EMIT_inp_file.write("source solar /home/zzbatmos/zzbatmos_common/Codes/libRadtran-2.0.4/data/solar_flux/kurudz_1.0nm.dat \n") 
        EMIT_inp_file.write("mol_abs_param reptran \n")

        EMIT_inp_file.write("rte_solver disort \n")

        EMIT_inp_file.write("aerosol_default \n")

        EMIT_inp_file.write("albedo {0:5.2f} \n".format(albedo_input))

        #  wvl   Exc_spheriod  Qe_spheriod  SSA_spheriod   g_spheriod
        # 0.4552   541.5262       2.4732       0.8392        0.8091    

        # for other wavelengths, will scale AOD with respect to Qe  

        EMIT_inp_file.write("wavelength {0:5.2f} {1:5.2f} \n".format(np.round(wl_input),np.round(wl_input)))
        #EMIT_inp_file.write("spline {0:5.2f} {1:5.2f} 0.01 \n".format(wl_input,wl_input))

        EMIT_inp_file.write("aerosol_modify ssa set {0:6.2f} \n".format(alb_input))
        EMIT_inp_file.write("aerosol_modify gg set {0:6.2f} \n".format(gg_input))
        if with_dust:
            EMIT_inp_file.write("aerosol_modify tau set {0:5.2f} \n".format(aod_input))
        else:
            EMIT_inp_file.write("aerosol_modify tau set {0:5.2f} \n".format(0.0))

        EMIT_inp_file.write("phi0 {0:6.2f} \n".format(0.0)) 
        EMIT_inp_file.write("sza {0:6.2f} \n".format(sza_input))
        EMIT_inp_file.write("day_of_year {0:3d}  \n".format(day_of_year_input))
        EMIT_inp_file.write("umu 0.1  0.12 0.14 0.16 0.18 0.2  0.22 0.24 0.26 0.28 0.3  0.32 0.34 0.36 0.38 0.4  0.42 0.44 0.46 0.48 0.5  0.52 0.54 0.56 0.58 0.6  0.62 0.64 0.66 0.68 0.7  0.72 0.74 0.76 0.78 0.8  0.82 0.84 0.86 0.88 0.9  0.92 0.94 0.96 0.98 1. \n") 
        #VZA 85, 80, 75, 70, 65, 60, 55, 50, 45, 40, 35, 30, 25, 20, 15, 10, 5, 0
        EMIT_inp_file.write("phi 0.   5.  10.  15.  20.  25.  30.  35.  40.  45.  50.  55.  60.  65. 70.  75.  80.  85.  90.  95. 100. 105. 110. 115. 120. 125. 130. 135. 140. 145. 150. 155. 160. 165. 170. 175. 180. \n")

        EMIT_inp_file.write("zout toa \n")
        #output_quantity reflectivity 
        #EMIT_inp_file.write("output_user lambda sza edir eglo edn eup enet esum\n")

        EMIT_inp_file.write("output_user lambda sza edir eglo edn eup enet esum fdir fglo fdn fup uu \n")
        #verbose 

 

In [ ]:
import subprocess
libradtran_datadir='/home/zzbatmos/zzbatmos_common/Codes/libRadtran-2.0.4/data/'
libradtran_exec='uvspec'
# #--- wavelength to run libradtran
# libradtran_wl = RAD.sensor_band_parameters['wavelengths'][G_idx]
# #--- surface albedo to run libradtran
# Ref_domain_avg = RFL.data.mean(axis=(0,1))
# libradtran_albedo = Ref_domain_avg[G_idx]

#---AOD550 
# AOD550_domain_avg =np.mean(MASK.data[:,:,5],axis=(0,1))
# print('AOD550',AOD550_domain_avg)

AOD550_domain_avg =MASK.data[pixel_yindex[1],pixel_xindex[1],5]
print('AOD550',AOD550_domain_avg)

#---solar zenith angle
atmos_file='/umbc/xfs1/zzbatmos-new/users/zzbatmos/Proposal/EMIT_2023/atmos_profile.dat'

day_of_year_input=(OBS.date-datetime.date(OBS.date.year,1,1))

print(day_of_year_input.days)

In [ ]:
print(OBS.data[:,:,4].mean())

### compute the fluxes and radiances for clear situation

In [ ]:
#with_dust=True
#write_EMIT_inp(True,libradtran_wl,libradtran_albedo,solar_zenith,AOD550_domain_avg,atmos_file)
#write_EMIT_inp(False,libradtran_wl,libradtran_albedo,solar_zenith,AOD550_domain_avg,atmos_file)
# libtratran_input = open("EMIT_dust_wl{0:05.2f}.INP".format(libradtran_wl),'r')
# libtratran_output = open("EMIT_dust_uu_wl{0:05.2f}.out".format(libradtran_wl),'wb')
# r=subprocess.run(libradtran_exec, stdin=libtratran_input,stdout=libtratran_output)
nwl = len(RAD.sensor_band_parameters['wavelengths'])
for iwl in range(nwl):
    libradtran_wl=np.round(RAD.sensor_band_parameters['wavelengths'][iwl]) 
    libradtran_albedo=np.nanmax([RFL.data[pixel_yindex[0],pixel_xindex[0],:][iwl],0.0])
    print(libradtran_wl,libradtran_albedo)
    #first compute the fluxes and radiances for clear situation
    write_EMIT_inp(False,libradtran_wl,
                   libradtran_albedo,
                   solar_zenith,
                   day_of_year_input.days,
                   AOD550_domain_avg/DOP.qe[DOP.wl0_index]*DOP.qe[iwl],
                   DOP.alb[iwl],DOP.gg[iwl],
                   atmos_file)
    libtratran_input = open("EMIT_clear_wl{0:05.2f}.INP".format(libradtran_wl),'r')
    libtratran_output = open("EMIT_clear_uu_wl{0:05.2f}.out".format(libradtran_wl),'wb')
    r=subprocess.run(libradtran_exec, 
                     stdin=libtratran_input,
                     stdout=libtratran_output,
                     stderr=subprocess.DEVNULL)
    
    #first compute the fluxes and radiances for w/ dust situation
    write_EMIT_inp(True,libradtran_wl,
                   libradtran_albedo,
                   solar_zenith,
                   day_of_year_input.days,
                   AOD550_domain_avg/DOP.qe[DOP.wl0_index]*DOP.qe[iwl],
                   DOP.alb[iwl],DOP.gg[iwl],
                   atmos_file)
    libtratran_input = open("EMIT_dust_wl{0:05.2f}.INP".format(libradtran_wl),'r')
    libtratran_output = open("EMIT_dust_uu_wl{0:05.2f}.out".format(libradtran_wl),'wb')
    r=subprocess.run(libradtran_exec, 
                     stdin=libtratran_input,
                     stdout=libtratran_output,
                     stderr=subprocess.DEVNULL)

#### read in the libradtran simulation of clean and dust laden case and plot the ADM

In [ ]:
nwl = len(RAD.sensor_band_parameters['wavelengths'])
edir_clear = np.zeros(nwl)
eup_clear  = np.zeros(nwl) 
edn_clear  = np.zeros(nwl)
fdn_clear  = np.zeros(nwl)
fup_clear  = np.zeros(nwl) 

edir_dust = np.zeros(nwl)
eup_dust  = np.zeros(nwl) 
edn_dust  = np.zeros(nwl)
fdn_dust = np.zeros(nwl)
fup_dust  = np.zeros(nwl) 


umu=np.array([0.1,  0.12, 0.14, 0.16, 0.18,
     0.2,  0.22, 0.24, 0.26, 0.28,
     0.3,  0.32, 0.34, 0.36, 0.38,
     0.4,  0.42, 0.44, 0.46, 0.48,
     0.5,  0.52, 0.54, 0.56, 0.58,
     0.6,  0.62, 0.64, 0.66, 0.68,
     0.7,  0.72, 0.74, 0.76, 0.78,
     0.8,  0.82, 0.84, 0.86, 0.88,
     0.9,  0.92, 0.94, 0.96, 0.98, 1. ])

phi=np.array([ 0.,   5.,  10.,  15.,  20.,
              25.,  30.,  35.,  40.,  45.,
              50.,  55.,  60.,  65., 70.,
              75.,  80.,  85.,  90.,  95.,
              100., 105., 110., 115., 120.,
              125., 130., 135., 140., 145.,
              150., 155., 160., 165., 170., 175., 180.])
nmu=len(umu)
nphi=len(phi)
ADM_dust = np.zeros([nwl,nmu,nphi])     

for iwl in range(nwl):
    libradtran_wl=np.round(RAD.sensor_band_parameters['wavelengths'][iwl]) #RAD.sensor_band_parameters['wavelengths'][iwl]
    libradtran_albedo=np.max([RFL.data.mean(axis=(0,1))[iwl],0.0])
    clear_out=np.loadtxt("ocean_pixel/EMIT_clear_uu_wl{0:05.2f}.out".format(libradtran_wl))

    dust_out=np.loadtxt("ocean_pixel/EMIT_dust_uu_wl{0:05.2f}.out".format(libradtran_wl))
    
    wl, sza , edir_clear[iwl] , eglo_clear, edn_clear[iwl], eup_clear[iwl], enet_clear, esum_clear,fdir, fglo, fdn_clear[iwl], fup_clear[iwl] = clear_out[0:12]
    wl, sza , edir_dust[iwl] , eglo_dust, edn_dust[iwl], eup_dust[iwl], enet_dust, esum_dust,fdir, fglo, fdn_dust[iwl], fup_dust[iwl]  = dust_out[0:12]
    ADM_dust[iwl,:,:]=dust_out[12:].reshape(nmu,nphi)/eup_dust[iwl]
    print(wl,edir_clear[iwl],eup_clear[iwl],fup_clear[iwl],dust_out[12:].shape,len(umu)*len(phi))
    # print('wl=',wl, 'sza=', sza ,'edir=', edir ,
    #       'eglo=',eglo, 'edn=', edn, 'eup=',eup, 'enet=',enet,
    #        'esum=',esum, 'fdir=', fdir, 'fglo=',fglo, 'fdn=',fdn, 'fup',fup)



In [ ]:
print(OBS.sensor_band_parameters['observation_bands'])
granule_mean_sza=OBS.data[:,:,4].mean()
granule_mean_vza=OBS.data[:,:,2].mean()
granule_mean_raz=OBS.data[:,:,1].mean()-OBS.data[:,:,3].mean()
print(granule_mean_sza,granule_mean_vza,granule_mean_raz)

In [ ]:
R_idx,G_idx,B_idx=36,22,10
print(RAD.sensor_band_parameters['wavelengths'][[R_idx,G_idx,B_idx]])
fig,ax=plt.subplots(figsize=[6,4],tight_layout=True)
ax.plot(RAD.sensor_band_parameters['wavelengths'],eup_clear,c='b')
ax.plot(RAD.sensor_band_parameters['wavelengths'],eup_dust,c='orange')
#ax.plot(RAD.sensor_band_parameters['wavelengths'],eup_dust-eup_clear,c='orange')

vaa = np.radians(np.linspace(0,180,nphi))
vza  = np.rad2deg(np.arccos(np.linspace(0.1,1,nmu))[::-1])#--- change the order of uu so that the zenith angle increases
r, theta = np.meshgrid(vza, vaa)

for idx in [R_idx,G_idx,B_idx]:
    
    fig, ax = plt.subplots(subplot_kw=dict(projection='polar'),figsize=[5,4],tight_layout=True)
    cf=ax.contourf(theta, r, np.transpose(ADM_dust[idx,::-1,:]*np.pi),
                   levels=20)
    plt.plot([np.radians(granule_mean_raz)],[granule_mean_vza],marker='*',c='r')
    plt.colorbar(cf,ax=ax,shrink=.6, pad=0.08)
    ax.set_thetamin(0)
    ax.set_thetamax(180)
    fig.savefig('ADM_example_wl{0:5.2f}.png'.format(RAD.sensor_band_parameters['wavelengths'][idx]),dpi=300)
#ax.set_title('Angular Distribution Model',fontsize='large')


# r, theta = np.meshgrid(vza, vaa)
# fig, ax = plt.subplots(subplot_kw=dict(projection='polar'))
# cf=ax.contourf(theta, r, np.transpose(ADM_dust[200,::-1,:]),
#                levels=20)
# plt.plot([np.radians(granule_mean_raz)],[granule_mean_vza],marker='*',c='r')
# plt.colorbar(cf,ax=ax,shrink=.6, pad=0.08)
# ax.set_thetamin(0)
# ax.set_thetamax(180)
# #ax.set_title('Angular Distribution Model',fontsize='large')



# # plt.show()

# # closure study to see if the integration of uu get back to upward flux eup
# uu=ADM_dust[DOP.wl0_index,::-1,:]*eup[DOP.wl0_index]
# uu_vza = np.zeros_like(vza)
# for i in range(len(vza)):
#     uu_vza[i]=np.trapz(uu[i,:],vaa)
# fig,ax=plt.subplots()
# ax.plot(vza,uu_vza)
# up_flux = np.trapz(uu_vza*np.cos(np.deg2rad(vza))*np.sin(np.deg2rad(vza)),np.deg2rad(vza))
# print(up_flux*2,eup[DOP.wl0_index])

# print(uu[10,10]/ADM[10,10])
# fig,ax=plt.subplots()
# ax.plot(mu,uu[:,0],c='r')
# ax.plot(mu,uu[:,10],c='g')
# ax.plot(mu,uu[:,-1],c='b')



# fig,ax=plt.subplots()
# ax.plot(vaa,uu[0,:],c='r')
# ax.plot(vaa,uu[10,:],c='g')
# ax.plot(vaa,uu[-1,:],c='b')


# flux_phi = np.zeros(nmu)
# for i in range(nmu):
#     flux_phi[i]=np.trapz(uu[i,:],vaa)
# fig,ax=plt.subplots()
# ax.plot(mu,flux_phi)
    
# flux= np.trapz(flux_phi*mu,mu)
# print('flux computed from uu',flux*2,eup)

# fig,ax=plt.subplots()
# cf=ax.pcolormesh(vaa,mu,uu/flux)
# plt.colorbar(cf,ax=ax)
# print(mu.shape)
# vza = np.rad2deg(np.arccos(mu))
# uu = np.transpose(uu[::-1,:])
# mu = mu[::-1]
# vza = vza[::-1]
# vza[0]=0.0

# r, theta = np.meshgrid(vza, vaa)
# #-- Plot... ------------------------------------------------
# fig, ax = plt.subplots(subplot_kw=dict(projection='polar'))
# cf=ax.contourf(theta, r, uu,levels=20)
# plt.colorbar(cf,ax=ax)
# ax.set_thetamin(0)
# ax.set_thetamax(180)

# fig,ax=plt.subplots()
# ax.plot(np.rad2deg(vaa),uu[:,0])

# ref_uu = np.pi*uu/edir

# vaa = np.radians(np.arange(0,180+1,10))
# mu  = np.arange(0.1,1.05,0.05)[::-1]
# vza = np.rad2deg(np.arccos(mu))[::-1]

# print(vaa.shape,vza.shape,uu.shape)
# fig,ax=plt.subplots()
# ax.plot(vza,uu[0,:])
    

In [ ]:
print(vza[-10])
print(phi[5])

In [ ]:
domain_avg_rfl = RFL.data[pixel_yindex[1],pixel_xindex[1],:]
import numpy.ma as ma
y=ma.masked_where(domain_avg_rfl <= 0, domain_avg_rfl)
#print(y)

S0_interp = np.interp(RAD.sensor_band_parameters['wavelengths'],kurudz_1nm.wl,kurudz_1nm.S0)
EMIT_converted_flux = RAD.data[pixel_yindex[1],pixel_xindex[1],:]*10.0/ADM_dust[:,43,5] # convert "uW/cm^2/SR/nm" to "mW/m^2/SR/nm"
# fig,ax=plt.subplots()
# ax.plot(umu,ADM_dust[G_idx,:,5])
# print(np.argmin(ADM_dust[G_idx,:,5]))
# print(umu[0])
# reflected_rad_no_atmos= y * (edir+edn) #*np.cos(np.radians(solar_zenith))

# ax.plot(kurudz_1nm.wl,kurudz_1nm.S0*np.cos(np.radians(solar_zenith)),c='m',lw=0.5,ls='dotted',label='solar insolation')

# ax.plot(RAD.sensor_band_parameters['wavelengths'],edir,c='b',label='edir')
# ax.plot(RAD.sensor_band_parameters['wavelengths'],edn,c='b',ls='dashed',label='edn')
fig,ax=plt.subplots(figsize=(6,3),tight_layout=True)
ax.plot(RAD.sensor_band_parameters['wavelengths'],EMIT_converted_flux,c='orange',label='Flux w/ dust covnerted from EMIT based on ADM')
ax.plot(RAD.sensor_band_parameters['wavelengths'],eup_clear,c='b',label='Flux w/o dust estimated from RTM ')

ax.fill_between(RAD.sensor_band_parameters['wavelengths'],
                EMIT_converted_flux,eup_clear,alpha=0.5,facecolor='orange')
ax.set_ylim([0,350])
ax.set_xlim([380,2500])
ax.legend(loc=0)
ax.set_xlabel('Wavelength [nm]')
ax.set_ylabel(r'Flux $[mW \ m^{-2} \ nm^{-1}]$')
ax.axvline(550,c='k',ls='dashed')
fig.savefig('flux_example.png',dpi=300)
# axr=ax.twinx()
# axr.set_ylabel( 'DRE $[W \ m^{-2} \ nm^{-1}]$',c='c')
# axr.plot(RAD.sensor_band_parameters['wavelengths'],eup_clear-eup_dust,c='c',ls='dashed',label='Spectrally-resolved DRE')
# axr.legend(loc=[0.4,0.25])
#axr.plot(RAD.sensor_band_parameters['wavelengths'],eup_dust-eup_clear)
#ax.plot(RAD.sensor_band_parameters['wavelengths'],reflected_rad_no_atmos,c='g',label='reflected')

print(np.trapz(kurudz_1nm.S0,kurudz_1nm.wl)/1000.0*np.cos(np.radians(solar_zenith)))
print(np.trapz(edir,RAD.sensor_band_parameters['wavelengths'])/1000.0)
print(np.trapz(eup_clear,RAD.sensor_band_parameters['wavelengths'])/1000.0)
print(np.trapz(EMIT_converted_flux,RAD.sensor_band_parameters['wavelengths'])/1000.0)
print(np.trapz(eup_dust,RAD.sensor_band_parameters['wavelengths'])/1000.0)

fig,ax=plt.subplots(figsize=(6,3),tight_layout=True)
#ax.plot(RAD.sensor_band_parameters['wavelengths'],EMIT_converted_flux,c='r')
#ax.plot(RAD.sensor_band_parameters['wavelengths'],eup_dust,c='r')
ax.plot(RAD.sensor_band_parameters['wavelengths'],eup_clear-EMIT_converted_flux,c='b')

#ax.set_ylim([0.0,0.4])


### test of sensitivity of ADM to AOD

In [ ]:
libradtran_wl=np.round(RAD.sensor_band_parameters['wavelengths'][G_idx]) 
libradtran_albedo=np.nanmax([RFL.data[pixel_yindex[1],pixel_xindex[1],:][G_idx],0.0])
print(libradtran_albedo)
AOD_values=np.arange(0.1,1.05,0.1)
for AOD in AOD_values:
    #first compute the fluxes and radiances for w/ dust situation
    write_EMIT_inp(True,libradtran_wl,
                   libradtran_albedo,
                   solar_zenith,
                   day_of_year_input.days,
                   AOD,
                   DOP.alb[G_idx],DOP.gg[G_idx],
                   atmos_file)
    libtratran_input = open("EMIT_dust_wl{0:05.2f}.INP".format(libradtran_wl),'r')
    libtratran_output = open("EMIT_dust_uu_AOD{0:05.2f}.out".format(AOD),'wb')
    r=subprocess.run(libradtran_exec, 
                     stdin=libtratran_input,
                     stdout=libtratran_output,
                     stderr=subprocess.DEVNULL)
    print(AOD)

In [ ]:
nAOD = len(AOD_values)
edir_dust = np.zeros(nAOD)
eup_dust  = np.zeros(nAOD) 
edn_dust  = np.zeros(nAOD)
fdn_dust = np.zeros(nAOD)
fup_dust  = np.zeros(nAOD) 

vaa = np.radians(np.linspace(0,180,nphi))
vza  = np.rad2deg(np.arccos(np.linspace(0.1,1,nmu))[::-1])#--- change the order of uu so that the zenith angle increases
r, theta = np.meshgrid(vza, vaa)

ADM_AOD = np.zeros([nAOD,nmu,nphi])     

for iAOD in range(nAOD):

    dust_out=np.loadtxt("EMIT_dust_uu_AOD{0:05.2f}.out".format(AOD_values[iAOD]))
    
    wl, sza , edir_dust[iAOD] , eglo_dust, edn_dust[iAOD], eup_dust[iAOD], enet_dust, esum_dust,fdir, fglo, fdn_dust[iAOD], fup_dust[iAOD]  = dust_out[0:12]
    ADM_AOD[iAOD,:,:]=dust_out[12:].reshape(nmu,nphi)/eup_dust[iAOD]

print(ADM_AOD[0,::-1,:].shape,theta.shape, r.shape)
fig, ax = plt.subplots(subplot_kw=dict(projection='polar'),
                           figsize=[5,4],tight_layout=True)
cf=ax.contourf(theta, r, np.transpose(ADM_AOD[0,::-1,:]),levels=20,
               vmin=0.2,vmax=0.4)
#plt.plot([np.radians(granule_mean_raz)],[granule_mean_vza],marker='*',c='r')
plt.colorbar(cf,ax=ax,shrink=.6, pad=0.08)
ax.set_thetamin(0)
ax.set_thetamax(180)

fig, ax = plt.subplots(subplot_kw=dict(projection='polar'),
                           figsize=[5,4],tight_layout=True)
cf=ax.contourf(theta, r, np.transpose(ADM_AOD[5,::-1,:]),levels=20,
               vmin=0.2,vmax=0.4)
#plt.plot([np.radians(granule_mean_raz)],[granule_mean_vza],marker='*',c='r')
plt.colorbar(cf,ax=ax,shrink=.6, pad=0.08)
ax.set_thetamin(0)
ax.set_thetamax(180)


fig, ax = plt.subplots(subplot_kw=dict(projection='polar'),
                           figsize=[5,4],tight_layout=True)
cf=ax.contourf(theta, r, np.transpose(ADM_AOD[-1,::-1,:]),levels=20,
               vmin=0.2,vmax=0.4)
#plt.plot([np.radians(granule_mean_raz)],[granule_mean_vza],marker='*',c='r')
plt.colorbar(cf,ax=ax,shrink=.6, pad=0.08)
ax.set_thetamin(0)
ax.set_thetamax(180)


fig, ax = plt.subplots(subplot_kw=dict(projection='polar'),
                           figsize=[5,4],tight_layout=True)
cf=ax.contourf(theta, r, np.transpose(ADM_AOD[-1,::-1,:]-ADM_AOD[0,::-1,:]),
               levels=20,cmap='bwr',vmin=-0.06,vmax=0.06)
#plt.plot([np.radians(granule_mean_raz)],[granule_mean_vza],marker='*',c='r')
plt.colorbar(cf,ax=ax,shrink=.6, pad=0.08)
ax.set_thetamin(0)
ax.set_thetamax(180)

In [ ]:
dust_out=np.loadtxt('EMIT_dust_ref.out',skiprows=2)
fig,ax=plt.subplots()
cf=ax.pcolormesh(dust_out[:,2:])
plt.colorbar(cf,ax=ax)
vaa = np.radians(np.arange(0,180+1,10))
mu  = dust_out[:,0]
vza = np.rad2deg(np.arccos(mu))[::-1]
ref = np.transpose(dust_out[:,2:][::-1,:])




# print(vza)
# print(vaa)
# print('ref',ref.shape,'vza',vza.shape,'vaa',vaa.shape)
# print(vza)


r, theta = np.meshgrid(vza, vaa)
# print('r',r)
weighting=np.cos(np.deg2rad(r))*np.sin(np.deg2rad(r))
F=np.trapz(np.trapz(ref*weighting, vaa , axis=0), vza, axis=0)
print(F*2)
print(np.mean(F*weighting))

#-- Plot... ------------------------------------------------
fig, ax = plt.subplots(subplot_kw=dict(projection='polar'))
cf=ax.contourf(theta, r, ref,levels=20)
plt.colorbar(cf,ax=ax)

ax.set_thetamin(0)
ax.set_thetamax(180)
plt.show()
print(uu)
int_flux = np.trapz(np.trapz(uu, y, axis=0), x, axis=0)

In [ ]:
class EMIT_Data(object):
    def __init__(self,L1B_url,L2_url):
        url_1 = L1B_url
        OBS_pattern = 'EMIT_L1B_OBS'
        RAD_pattern = 'EMIT_L1B_RAD'

        MASK_pattern = 'EMIT_L2A_MASK'
        RFL_pattern = 'EMIT_L2A_RFL_'
        RFL_UNCERT_pattern = 'EMIT_L2A_RFLUNCERT'

        for f in s3_fs.listdir(url_1):
            if(f['Key'].endswith('.nc')):
                #print(f['Key'])
                if OBS_pattern in f['Key']:
                    print(f['Key'])
                    with s3_fs.open(s3_fs.open(f['Key']),'rb') as f1:
                        with xr.open_dataset(f1,engine='h5netcdf',chunks={}) as ds:
                            self.obs=ds['obs'].to_numpy()
                            #print('self.obs',self.obs)
                        # with xr.open_dataset(f1,engine='h5netcdf',group='location',chunks={}) as ds:
                        #     self.obs_location=ds['location'].to_numpy()
                        #OBS_data.close()
                        # OBS_data_location = xr.open_dataset(f1,group='location',engine='h5netcdf')
                        # self.obs_location=OBS_data_location.location
                        # OBS_data_location.close()
                        # f2.close()
                        # f3=s3_fs.open(s3_fs.open(f['Key']))
                        # OBS_data_sbp = xr.open_dataset(f3,group='sensor_band_parameters',engine='h5netcdf') 
                        # f3.close()
                elif RAD_pattern in f['Key']:
                    print(f['Key'])
                    with s3_fs.open(s3_fs.open(f['Key']),'rb') as f1:
                        with xr.open_dataset(f1,engine='h5netcdf') as ds: 
                            self.rad = ds.radiance.to_numpy()
                        #RAD_data.close()
                        # f4.close()
                        # f5=s3_fs.open(s3_fs.open(f['Key']))
                        # RAD_data_location = xr.open_dataset(f5,group='location',engine='h5netcdf') 
                        # f5.close()
                        # f6=s3_fs.open(s3_fs.open(f['Key']))
                        # RAD_data_sbp = xr.open_dataset(f6,group='sensor_band_parameters',engine='h5netcdf') 
                        # f6.close()
                        #E3=EMIT_L1B_OBS('s3://'+f['Key'])

        url_2 = L2_url
        for f in s3_fs.listdir(url_2):
            if(f['Key'].endswith('.nc')):
                #print(f['Key'])
                if MASK_pattern in f['Key']:
                    print('MASK,',f['Key'])
                    with s3_fs.open(s3_fs.open(f['Key']),'rb') as f1:
                         with xr.open_dataset(f1,engine='h5netcdf') as ds:
                            self.mask=ds.mask.to_numpy()
                        #MASK_data.close()
                elif (RFL_pattern in f['Key']) and (RFL_UNCERT_pattern in f['Key']):
                    print('RFL,',f['Key'])
                    with s3_fs.open(s3_fs.open(f['Key'])) as f1:
                        with xr.open_dataset(f1,engine='h5netcdf') as ds:
                            #print(RFL_unc_data)
                            self.rfl_unc=ds.reflectance_uncertainty.to_numpy()
                            #RFL_unc_data.close()
                elif RFL_pattern in f['Key']:
                    print('RFL2,',f['Key'])
                    with s3_fs.open(s3_fs.open(f['Key']),'rb') as f1:
                        with xr.open_dataset(f1,engine='h5netcdf') as ds:
                            #print(RFL_data)
                            self.rfl=ds.reflectance.to_numpy()
                            #RFL_data.close()
                    # f1.close()
                    # f1=s3_fs.open(s3_fs.open(f['Key']))
                    # RFL_data_location = xr.open_dataset(f1,group='location',engine='h5netcdf') 
                    # f1.close()
                    # f1=s3_fs.open(s3_fs.open(f['Key']))
                    # RFL_data_sbp = xr.open_dataset(f1,group='sensor_band_parameters',engine='h5netcdf') 
                    # f1.close()
    # OBS_data.close()
    # OBS_data_location.close()
    # OBS_data_sbp.close()
    # RAD_data.close()
    # RAD_data_location.close()
    # RAD_data_sbp.close()
    # RFL_data.close()
    # RFL_data_location.close()
    # RFL_data_sbp.close()
    # MASK_data.close()
    # RFL_unc_data.close()
    # return OBS_data,OBS_data_location,OBS_data_sbp,\
    #        RAD_data,RAD_data_location,RAD_data_sbp,\
    #        RFL_data,RFL_data_location,RFL_data_sbp,MASK_data,RFL_unc_data



# statistical analysis of the EMIT AOD data

In [ ]:
import glob
AOD550_index=5
AOD550_all = np.array([])
MASK_list=glob.glob(EMIT_data_path+MASK_pattern+'*.nc')
for i in range(len(MASK_list)):
    M=EMIT_Data(MASK_list[i],'mask','L2A_MASK')
    AOD550_all=np.append(AOD550_all, M.data[:,:,AOD550_index].flatten())
    #print(MASK_list[i])


In [ ]:
np.savez_compressed('AOD550_all',AOD550_all=AOD550_all)

In [ ]:
d=np.load('AOD550_all.npz')
print(d['AOD550_all'].shape)

In [ ]:
print(AOD550_all.shape)
import numpy.ma as ma
AOD550_all_masked= ma.masked_where(AOD550_all<  0, AOD550_all)
AOD_550_bins=np.linspace(0.01, 0.7, 100)
fig,ax=plt.subplots(figsize=[10,3])
ax.hist(AOD550_all_masked,bins=AOD_550_bins,density=True)
ax.set_xlabel('AOD @ 550 nm')
ax.set_ylabel('PDF')

In [ ]:
print(np.mean(AOD550_all_masked),np.median(AOD550_all_masked))

In [ ]:
import numpy.ma as ma

AOD_550_bins=np.linspace(0.0, 0.7, 100)
fig,ax=plt.subplots(figsize=[10,3])
ax.hist(d['AOD550_all'],bins=AOD_550_bins,density=True)
ax.set_xlabel('AOD @ 550 nm')
ax.set_ylabel('PDF')

In [ ]:
print(np.mean(d['AOD550_all']))

## Geolocation test

### read in all the latitude and longitude from data

In [ ]:
EMIT_data_path='/home/zzbatmos/zzbatmos_common/Data/EMIT/'

OBS_pattern = 'EMIT_L1B_OBS'
MASK_pattern = 'EMIT_L2A_MASK'

L1B_OBS_files=glob.glob(EMIT_data_path+OBS_pattern+'*.nc')
L2A_MASK_files=glob.glob(EMIT_data_path+MASK_pattern+'*.nc')
#print(L2A_MASK_files)
ncases=0 #len(case_files)

EMIT_geolocation_cases={'L1B_OBS_filelist':[],
                        'L2A_MASK_filelist':[],
                        'granule_lat_all':[],
                        'granule_lon_all':[],
                        'AOD500_all':[],
                        'cloudmask_all':[],
                        'cirrusmask_all':[],
                        'dilatedcloudmask_all':[],
                        'date_all':[]}

for L1B_file in L1B_OBS_files:
    p=L1B_file.find('OBS_001_')
    #print(L1B_file)
    time_stamp = L1B_file[p+8:p+8+15]
    #print(time_stamp)
    filelist = [i for i in L2A_MASK_files if time_stamp in i]
    #print(filelist)
    if len(filelist) ==0:
        continue
    else:
        ncases+=1
        if (ncases%100==0): print(ncases)
        OBS=EMIT_Data(L1B_file,'obs','L1B_OBS','L1B')
        MASK=EMIT_Data(filelist[0],'mask','L2A_MASK','L2A')

        EMIT_geolocation_cases['L1B_OBS_filelist'].append(L1B_file)
        EMIT_geolocation_cases['L2A_MASK_filelist'].append(filelist[0])
        EMIT_geolocation_cases['granule_lat_all'].append(OBS.location['lat'])
        EMIT_geolocation_cases['granule_lon_all'].append(OBS.location['lon'])
        EMIT_geolocation_cases['AOD500_all'].append(MASK.data[:,:,5])
        EMIT_geolocation_cases['cloudmask_all'].append(MASK.data[:,:,0])
        EMIT_geolocation_cases['cirrusmask_all'].append(MASK.data[:,:,1])
        EMIT_geolocation_cases['dilatedcloudmask_all'].append(MASK.data[:,:,4])
        EMIT_geolocation_cases['date_all'].append(OBS.date)


with open('EMIT_geolocation_cases.pkl', 'wb') as f:
    pickle.dump(EMIT_geolocation_cases, f)

In [ ]:
!pwd
%cd /umbc/xfs1/zzbatmos/users/zzbatmos/Proposal/EMIT_2023/

In [ ]:
if os.path.isfile('EMIT_geolocation_cases.pkl'):
    EMIT_geolocation_cases=pickle.load(open('EMIT_geolocation_cases.pkl','rb'))


In [ ]:
lat_grids=np.arange(10,36,1)
lon_grids=np.arange(-10,35,1)
ncases = len(EMIT_geolocation_cases['AOD500_all'])
print(ncases)
L1B_OBS_gridded = [{'L1B_OBS_files':[],'L2A_MASK_files':[]} for i in range(len(lat_grids)*len(lon_grids))]
L1B_OBS_gridded=np.array(L1B_OBS_gridded).reshape(len(lat_grids),len(lon_grids))
print(L1B_OBS_gridded[0,0])

### location all the laitude and longitude into 1x1 degree box

In [ ]:
# generate some latitude and longitude grids

Max_AOD_threshold = 2.0
AOD_grids = np.arange(0,Max_AOD_threshold+0.01,0.01)
nlat,nlon,nAOD=len(lat_grids),len(lon_grids),len(AOD_grids)
lons_mesh, lats_mesh = np.meshgrid(lon_grids, lat_grids)

# put all the latitude and longitude into boxes
all_pixels_lat_lon_AOD_total_counts = np.zeros([len(lat_grids),len(lon_grids),len(AOD_grids)])
cloudfree_pixels_lat_lon_AOD_total_counts = np.zeros([len(lat_grids),len(lon_grids),len(AOD_grids)])
datafile_gridded = [{'L1B_OBS_files':[],'L2A_MASK_files':[]} for i in range(len(lat_grids)*len(lon_grids))]
datafile_gridded = np.array(datafile_gridded).reshape(nlat,nlon)

start_date = datetime.date(2023,4,1)
end_date = datetime.date(2023,4,30)
for icase in range(ncases):
    if (EMIT_geolocation_cases['date_all'][icase]<start_date) or \
       (EMIT_geolocation_cases['date_all'][icase]>end_date):
        continue
    print(EMIT_geolocation_cases['date_all'][icase])    
    if icase%100==0:print('processing',icase)
    nx,ny=EMIT_geolocation_cases['cloudmask_all'][icase].shape
    cloudfree_pixels_index = (EMIT_geolocation_cases['cloudmask_all'][icase]==0) & \
                             (EMIT_geolocation_cases['cirrusmask_all'][icase]==0) & \
                             (EMIT_geolocation_cases['dilatedcloudmask_all'][icase]==0) & \
                            (EMIT_geolocation_cases['AOD500_all'][icase]<= Max_AOD_threshold)
    
    cloudfree_pixels_lat_index=np.searchsorted(lat_grids,EMIT_geolocation_cases['granule_lat_all'][icase][cloudfree_pixels_index])
    cloudfree_pixels_lon_index=np.searchsorted(lon_grids,EMIT_geolocation_cases['granule_lon_all'][icase][cloudfree_pixels_index])
    cloudfree_pixels_AOD_index=np.searchsorted(AOD_grids,EMIT_geolocation_cases['AOD500_all'][icase][cloudfree_pixels_index])
    
    cloudfree_clean_pixels_lat_lon_AOD_combined=np.array([cloudfree_pixels_lat_index.flatten(),
                                                          cloudfree_pixels_lon_index.flatten(),
                                                          cloudfree_pixels_AOD_index.flatten()])
    cloudfree_clean_pixels_lat_lon_combined=np.array([cloudfree_pixels_lat_index.flatten(),
                                                          cloudfree_pixels_lon_index.flatten()])
    
    #===gelocate all the cloud free and clean pixeles
    lat_lon_AOD_unique,cunt=np.unique(cloudfree_clean_pixels_lat_lon_AOD_combined,axis=1,return_counts=True)
    _,nq=lat_lon_AOD_unique.shape
    for ipair in range(nq):
        cloudfree_pixels_lat_lon_AOD_total_counts[lat_lon_AOD_unique[0,ipair],
                                                  lat_lon_AOD_unique[1,ipair],
                                                  lat_lon_AOD_unique[2,ipair]]+=cunt[ipair]
    #===gelocate all the cloud free and clean pixeles
    lat_lon_unique,cunt=np.unique(cloudfree_clean_pixels_lat_lon_combined,axis=1,return_counts=True)
    _,nq=lat_lon_unique.shape
    for ipair in range(nq):
        ilat = lat_lon_unique[0,ipair]
        ilon = lat_lon_unique[1,ipair]
        datafile_gridded[ilat,ilon]['L1B_OBS_files'].append(EMIT_geolocation_cases['L1B_OBS_filelist'][icase])
        datafile_gridded[ilat,ilon]['L2A_MASK_files'].append(EMIT_geolocation_cases['L2A_MASK_filelist'][icase])

  


In [ ]:
datafile_gridded[15,24]
print(lat_grids[15],lon_grids[24])

In [ ]:
AOD_10percentile=np.zeros([len(lat_grids),len(lon_grids)])
AOD_50percentile=np.zeros([len(lat_grids),len(lon_grids)])

for i in range(len(lat_grids)):
    for j in range(len(lon_grids)):
        if cloudfree_pixels_lat_lon_AOD_total_counts[i,j,:].max()<=0: 
            AOD_10percentile[i,j]=np.nan
            AOD_50percentile[i,j]=np.nan

            continue
        AOD_CDF=np.cumsum(cloudfree_pixels_lat_lon_AOD_total_counts[i,j,:])
        AOD_CDF/=AOD_CDF[-1]
        percentile10=np.argmin(np.abs(AOD_CDF-0.1))
        percentile50=np.argmin(np.abs(AOD_CDF-0.5))
        AOD_10percentile[i,j]=AOD_grids[percentile10]
        AOD_50percentile[i,j]=AOD_grids[percentile50]


In [ ]:
fig,ax=plt.subplots()
cf=ax.pcolormesh(AOD_10percentile,vmin=0.0,vmax=0.7)
plt.colorbar(cf,extend='max')
print(np.nanmax(AOD_10percentile))
# fig,ax=plt.subplots()
# ax.plot(AOD_grids,cloudfree_pixels_lat_lon_AOD_total_counts.sum(axis=(0,1)))
# ax.set_xlim([0,1])
# cdf=np.cumsum(cloudfree_pixels_lat_lon_AOD_total_counts.sum(axis=(0,1)))

# fig,ax=plt.subplots()
# ax.plot(AOD_grids,cdf/cdf[-1])
# ax.set_xlim([0,1])

In [ ]:
fig = plt.figure(figsize=(6,6))

ax = fig.add_subplot(1,1,1, projection=crs.Mercator())

ax.stock_img()
ax.coastlines()
ax.add_feature(cfeature.BORDERS)

lat1, lon1, lat2, lon2 = -15, 35, 0, 35
ax.set_extent([lat1, lon1, lat2, lon2], crs=crs.PlateCarree())
plt.show()

In [ ]:
fig = plt.figure(figsize=(10,8))

ax = fig.add_subplot(1,1,1, projection=crs.Mercator())

ax.set_global()

ax.stock_img()
ax.coastlines()
ax.add_feature(cfeature.BORDERS)

lat1, lon1, lat2, lon2 = -15, 35, 0, 35
ax.set_extent([lat1, lon1, lat2, lon2], crs=crs.PlateCarree())
for i in range(1):#len(granule_lat_all)
    plt.scatter(x=granule_lon_all[i], y=granule_lat_all[i],
                color="red",
                s=0.2,
                alpha=0.5,
                transform=crs.PlateCarree()) ## Important
    
gl = ax.gridlines(crs=crs.PlateCarree(), 
                  draw_labels=True, 
                  linewidth = 1)
gl.xlocator = mticker.FixedLocator(np.arange(-15,35,1))
gl.ylocator = mticker.FixedLocator(np.arange(-20,40,1))
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER

plt.show()

In [ ]:
fig = plt.figure(figsize=(8,4),tight_layout=True)

ax = fig.add_subplot(1,1,1, projection=crs.Mercator())

ax.set_global()

ax.stock_img()
ax.coastlines()
ax.add_feature(cfeature.BORDERS)

lat1, lon1, lat2, lon2 = -10, 35, 13, 35
cunt=cloudfree_pixels_lat_lon_AOD_total_counts.sum(axis=2)
cunt[cunt<=0]=np.nan
ax.set_extent([lat1, lon1, lat2, lon2],crs=crs.PlateCarree())
cf=ax.pcolormesh(lons_mesh, lats_mesh,cunt ,
                transform=crs.PlateCarree(),
                cmap='viridis',alpha=0.5)
    
gl = ax.gridlines(crs=crs.PlateCarree(), 
                  draw_labels=True, 
                  linewidth = 1)
gl.xlocator = mticker.FixedLocator(np.arange(-15,35,1))
gl.ylocator = mticker.FixedLocator(np.arange(-20,40,1))
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER
plt.colorbar(cf,ax=ax,fraction=.08, pad=0.08, shrink=0.5, aspect=12,label='# of pixels')
fig.savefig('number_of_pixel.png',dpi=300)


fig = plt.figure(figsize=(8,4),tight_layout=True)

ax = fig.add_subplot(1,1,1, projection=crs.Mercator())

ax.set_global()

ax.stock_img()
ax.coastlines()
ax.add_feature(cfeature.BORDERS)

lat1, lon1, lat2, lon2 = -10, 35, 13, 35
ax.set_extent([lat1, lon1, lat2, lon2], crs=crs.PlateCarree())
cf=ax.pcolormesh(lons_mesh, lats_mesh, AOD_10percentile,
                transform=crs.PlateCarree(),
                cmap='viridis',alpha=0.5)
    
gl = ax.gridlines(crs=crs.PlateCarree(), 
                  draw_labels=True, 
                  linewidth = 1)
gl.xlocator = mticker.FixedLocator(np.arange(-15,35,1))
gl.ylocator = mticker.FixedLocator(np.arange(-20,40,1))
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER
plt.colorbar(cf,ax=ax,fraction=.08, pad=0.08, shrink=0.5, aspect=12,label='10 percentile AOD')
fig.savefig('10_percentile_AOD.png',dpi=300)


fig = plt.figure(figsize=(8,4),tight_layout=True)

ax = fig.add_subplot(1,1,1, projection=crs.Mercator())

ax.set_global()

ax.stock_img()
ax.coastlines()
ax.add_feature(cfeature.BORDERS)

lat1, lon1, lat2, lon2 = -10, 35, 13, 35
ax.set_extent([lat1, lon1, lat2, lon2], crs=crs.PlateCarree())
cf=ax.pcolormesh(lons_mesh, lats_mesh, AOD_50percentile,
                transform=crs.PlateCarree(),
                cmap='viridis',alpha=0.5)
    
gl = ax.gridlines(crs=crs.PlateCarree(), 
                  draw_labels=True, 
                  linewidth = 1)
gl.xlocator = mticker.FixedLocator(np.arange(-15,35,1))
gl.ylocator = mticker.FixedLocator(np.arange(-20,40,1))
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER
plt.colorbar(cf,ax=ax,fraction=.08, pad=0.08, shrink=0.5, aspect=12,label='Media AOD')
fig.savefig('50_percentile_AOD.png',dpi=300)


In [ ]:
lat_0,lon_0=25,1
#lat_0,lon_0=18,-5
lat_0_idx=(np.searchsorted(lat_grids,lat_0))
lon_0_idx=(np.searchsorted(lon_grids,lon_0))
print(lat_grids[lat_0_idx],lon_grids[lon_0_idx])

In [ ]:
cloudfree_pixels_lat_lon_AOD_total_counts[lat_0_idx,lon_0_idx,:].sum()

In [ ]:
fig,ax=plt.subplots()
ax.plot(AOD_grids,cloudfree_pixels_lat_lon_AOD_total_counts[lat_0_idx,lon_0_idx,:])
ax.set_xlim([0,1])
print(AOD_10percentile[lat_0_idx,lon_0_idx])

### analysis the L1B_RAD and L2_RFL in a selected grid box

In [ ]:
print((datafile_gridded[lat_0_idx,lon_0_idx]['L1B_OBS_files']))

In [ ]:
# list of L1 and L2 data files in the selected grid
L1B_RAD_filelist=['EMIT_L1B_RAD_001_20230417T144248_2310710_014.nc',
                  'EMIT_L1B_RAD_001_20230407T094144_2309707_028.nc',
                  'EMIT_L1B_RAD_001_20230407T094132_2309707_027.nc',
                  'EMIT_L1B_RAD_001_20230403T111627_2309308_027.nc']
L2A_RFL_filelist=['EMIT_L2A_RFL_001_20230417T144248_2310710_014.nc',
                  'EMIT_L2A_RFL_001_20230407T094144_2309707_028.nc',
                  'EMIT_L2A_RFL_001_20230407T094132_2309707_027.nc',
                  'EMIT_L2A_RFL_001_20230403T111627_2309308_027.nc']
L2A_MASK_filelist=['EMIT_L2A_MASK_001_20230417T144248_2310710_014.nc',
                   'EMIT_L2A_MASK_001_20230407T094144_2309707_028.nc',
                   'EMIT_L2A_MASK_001_20230407T094132_2309707_027.nc',
                   'EMIT_L2A_MASK_001_20230403T111627_2309308_027.nc']

In [ ]:
# read in all the L1 and L2 data
OBS_pattern = 'EMIT_L1B_OBS'
RAD_pattern = 'EMIT_L1B_RAD'

MASK_pattern = 'EMIT_L2A_MASK'
RFL_pattern = 'EMIT_L2A_RFL_'
RFL_UNCERT_pattern = 'EMIT_L2A_RFLUNCERT'

MIN_pattern = 'EMIT_L2B_MIN_'
MIN_UNCERT_pattern = 'EMIT_L2B_MINUNCERT'

EMIT_data_path='/home/zzbatmos/zzbatmos_common/Data/EMIT/'
RAD_in_grid=[]
RFL_in_grid=[]
MASK_in_grid=[]
for i in range(len(datafile_gridded[lat_0_idx,lon_0_idx]['L1B_OBS_files'])):
    RAD_in_grid.append(EMIT_Data(EMIT_data_path+L1B_RAD_filelist[i],'radiance','L1B_RAD','L1B'))
    RFL_in_grid.append(EMIT_Data(EMIT_data_path+L2A_RFL_filelist[i],'reflectance','L2A_RFL','L2A'))
    MASK_in_grid.append(EMIT_Data(EMIT_data_path+L2A_MASK_filelist[i],'mask','L2A_MASK','L2A'))
    

In [ ]:
# find all the cloud free pixels in the grid
for i in range(len(datafile_gridded[lat_0_idx,lon_0_idx]['L1B_OBS_files'])):
    lat_index = np.searchsorted(lat_grids,MASK_in_grid[i].location['lat'])
    lon_index = np.searchsorted(lon_grids,MASK_in_grid[i].location['lon'])
    cloudfree_pixels_in_grid = (MASK_in_grid[i].data[:,:,0]==0) & \
                              (MASK_in_grid[i].data[:,:,1]==0) & \
                              (MASK_in_grid[i].data[:,:,4]==0) & \
                              (MASK_in_grid[i].data[:,:,5]<=Max_AOD_threshold) & \
                              (lat_index==lat_0_idx) & (lon_index==lon_0_idx)
    
    cloudfree_low_AOD_pixels_in_grid = (cloudfree_pixels_in_grid) & (MASK_in_grid[i].data[:,:,5]<=AOD_10percentile[lat_0_idx,lon_0_idx])
    cloudfree_high_AOD_pixels_in_grid = (cloudfree_pixels_in_grid) & (MASK_in_grid[i].data[:,:,5]>AOD_10percentile[lat_0_idx,lon_0_idx])
    print('number of cloud free pixels with low AOD',cloudfree_low_AOD_pixels_in_grid.sum())
    RFL_granule_low_AOD=np.array([RFL_in_grid[i].data[:,:,j][cloudfree_low_AOD_pixels_in_grid] for j in range(285)])
    RFL_granule_high_AOD=np.array([RFL_in_grid[i].data[:,:,j][cloudfree_high_AOD_pixels_in_grid] for j in range(285)])
    RAD_granule_low_AOD=np.array([RAD_in_grid[i].data[:,:,j][cloudfree_low_AOD_pixels_in_grid] for j in range(285)])
    RAD_granule_high_AOD=np.array([RAD_in_grid[i].data[:,:,j][cloudfree_high_AOD_pixels_in_grid] for j in range(285)])
    if i==0:
        RFL_cloudfree_low_AOD_pixels_in_grid= RFL_granule_low_AOD
        RFL_cloudfree_high_AOD_pixels_in_grid= RFL_granule_high_AOD
        RAD_cloudfree_low_AOD_pixels_in_grid= RAD_granule_low_AOD
        RAD_cloudfree_high_AOD_pixels_in_grid= RAD_granule_high_AOD
    else:
        RFL_cloudfree_low_AOD_pixels_in_grid=np.append(RFL_cloudfree_low_AOD_pixels_in_grid,RFL_granule_low_AOD,axis=1)
        RFL_cloudfree_high_AOD_pixels_in_grid=np.append(RFL_cloudfree_high_AOD_pixels_in_grid,RFL_granule_high_AOD,axis=1)
        RAD_cloudfree_low_AOD_pixels_in_grid=np.append(RAD_cloudfree_low_AOD_pixels_in_grid,RFL_granule_low_AOD,axis=1)
        RAD_cloudfree_high_AOD_pixels_in_grid=np.append(RAD_cloudfree_high_AOD_pixels_in_grid,RFL_granule_high_AOD,axis=1)
                                                       
    


In [ ]:
RFL_mean_lowAOD  = RFL_cloudfree_low_AOD_pixels_in_grid.mean(axis=1)
RFL_std_lowAOD  = RFL_cloudfree_low_AOD_pixels_in_grid.std(axis=1)
RFL_mean_highAOD = RFL_cloudfree_high_AOD_pixels_in_grid.mean(axis=1)
RFL_std_highAOD  = RFL_cloudfree_high_AOD_pixels_in_grid.std(axis=1)
gap=RFL_mean_lowAOD<=0
RFL_mean_lowAOD[gap]= np.nan
RFL_std_lowAOD[gap]=np.nan
RFL_mean_highAOD[gap]= np.nan
RFL_std_highAOD[gap]=np.nan

fig,ax=plt.subplots(figsize=[8,3],tight_layout=True)
ax.plot(RAD_in_grid[0].sensor_band_parameters['wavelengths'],RFL_mean_lowAOD)
ax.fill_between(RAD_in_grid[0].sensor_band_parameters['wavelengths'],
                RFL_mean_lowAOD-RFL_std_lowAOD,RFL_mean_lowAOD+RFL_std_lowAOD,
                alpha=0.3, facecolor='g')

ax.plot(RAD_in_grid[0].sensor_band_parameters['wavelengths'],RFL_mean_highAOD)
ax.fill_between(RAD_in_grid[0].sensor_band_parameters['wavelengths'],
                RFL_mean_highAOD-RFL_std_highAOD,RFL_mean_highAOD+RFL_std_highAOD,
                alpha=0.3, facecolor='m')


RAD_mean_lowAOD  = RAD_cloudfree_low_AOD_pixels_in_grid.mean(axis=1)
RAD_std_lowAOD  = RAD_cloudfree_low_AOD_pixels_in_grid.std(axis=1)
RAD_mean_highAOD = RAD_cloudfree_high_AOD_pixels_in_grid.mean(axis=1)
RAD_std_highAOD  = RAD_cloudfree_high_AOD_pixels_in_grid.std(axis=1)
gap=RAD_mean_lowAOD<=0
RAD_mean_lowAOD[gap]= np.nan
RAD_std_lowAOD[gap]=np.nan
RAD_mean_highAOD[gap]= np.nan
RAD_std_highAOD[gap]=np.nan

fig,ax=plt.subplots(figsize=[8,3],tight_layout=True)
ax.plot(RAD_in_grid[0].sensor_band_parameters['wavelengths'],RAD_mean_lowAOD,c='g',lw=2,ls='solid')
# ax.fill_between(RAD_in_grid[0].sensor_band_parameters['wavelengths'],
#                 RAD_mean_lowAOD-RAD_std_lowAOD,RAD_mean_lowAOD+RAD_std_lowAOD,
#                 alpha=0.3, facecolor='g')
ax.plot(RAD_in_grid[0].sensor_band_parameters['wavelengths'],RAD_mean_highAOD,c='m',lw=2,ls='dashed')
# ax.fill_between(RAD_in_grid[0].sensor_band_parameters['wavelengths'],
#                 RAD_mean_highAOD-RAD_std_highAOD,RAD_mean_highAOD+RAD_std_highAOD,
#                 alpha=0.3, facecolor='m')

In [ ]:
print(RFL_mean_lowAOD)